In [20]:
import pandas as pd
import numpy as np

In [21]:
movies = pd.read_csv("../data/raw/movies.csv")
ratings = pd.read_csv("../data/raw/ratings.csv")
tags = pd.read_csv("../data/raw/tags.csv")

In [22]:
df = ratings.merge(movies, on="movieId", how="left")

df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [23]:
df["rating_date"] = pd.to_datetime(df["timestamp"], unit="s")

df.head()

,userId,movieId,rating,timestamp,title,genres,rating_date
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2000-07-30 18:45:03
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,2000-07-30 18:20:47
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,2000-07-30 18:37:04
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,2000-07-30 19:03:35
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,2000-07-30 18:48:51


In [24]:
df["rating_year"] = df["rating_date"].dt.year
df["rating_month"] = df["rating_date"].dt.month

df.head()

,userId,movieId,rating,timestamp,title,genres,rating_date,rating_year,rating_month
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2000-07-30 18:45:03,2000,7
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,2000-07-30 18:20:47,2000,7
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,2000-07-30 18:37:04,2000,7
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,2000-07-30 19:03:35,2000,7
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,2000-07-30 18:48:51,2000,7


In [25]:
df["release_year"] = (
    df["title"]
      .str.extract(r"\((\d{4})\)")
      .astype(float)
)

df[["title", "release_year"]].head()

,title,release_year
0,Toy Story (1995),1995.0
1,Grumpier Old Men (1995),1995.0
2,Heat (1995),1995.0
3,Seven (a.k.a. Se7en) (1995),1995.0
4,"Usual Suspects, The (1995)",1995.0


In [26]:
df["title"] = df["title"].str.replace(
    r"\s*\(\d{4}\)",
    "",
    regex=True
)

df[["title", "release_year"]].head()

,title,release_year
0,Toy Story,1995.0
1,Grumpier Old Men,1995.0
2,Heat,1995.0
3,Seven (a.k.a. Se7en),1995.0
4,"Usual Suspects, The",1995.0


In [27]:
df.to_csv("../data/processed/movies_step1.csv", index=False)

print("Feature Engineering Step 1 Complete!")

Feature Engineering Step 1 Complete!


In [28]:
genre_df = df.copy()

genre_df["genres"] = genre_df["genres"].str.split("|")

genre_df = genre_df.explode("genres")

genre_df.head()

,userId,movieId,rating,timestamp,title,genres,rating_date,rating_year,rating_month,release_year
0,1,1,4.0,964982703,Toy Story,Adventure,2000-07-30 18:45:03,2000,7,1995.0
0,1,1,4.0,964982703,Toy Story,Animation,2000-07-30 18:45:03,2000,7,1995.0
0,1,1,4.0,964982703,Toy Story,Children,2000-07-30 18:45:03,2000,7,1995.0
0,1,1,4.0,964982703,Toy Story,Comedy,2000-07-30 18:45:03,2000,7,1995.0
0,1,1,4.0,964982703,Toy Story,Fantasy,2000-07-30 18:45:03,2000,7,1995.0


In [29]:
movie_stats = (
    df.groupby(["movieId", "title"])
      .agg(
          avg_rating=("rating", "mean"),
          num_ratings=("rating", "count"),
          rating_std=("rating", "std")
      )
      .reset_index()
)

movie_stats.head()

,movieId,title,avg_rating,num_ratings,rating_std
0,1,Toy Story,3.920930,215,0.834859
1,2,Jumanji,3.431818,110,0.881713
2,3,Grumpier Old Men,3.259615,52,1.054823
3,4,Waiting to Exhale,2.357143,7,0.852168
4,5,Father of the Bride Part II,3.071429,49,0.907148


In [30]:
user_stats = (
    df.groupby("userId")
      .agg(
          avg_rating_given=("rating", "mean"),
          num_movies_rated=("rating", "count")
      )
      .reset_index()
)

user_stats.head()

,userId,avg_rating_given,num_movies_rated
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


In [31]:
user_genres = ratings.merge(
    genre_df[["movieId", "genres"]],
    on="movieId"
)

user_genres.head()
favorite_genre = (
    user_genres
    .groupby(["userId", "genres"])
    .size()
    .reset_index(name="count")
)

favorite_genre.head()
favorite_genre = (
    favorite_genre
    .sort_values("count", ascending=False)
    .drop_duplicates("userId")
)

favorite_genre.head()

,userId,genres,count
6805,414,Drama,30965
7786,474,Drama,25501
9816,599,Drama,25036
6252,380,Action,22373
1125,68,Comedy,22292


In [32]:
user_stats = user_stats.merge(
    favorite_genre[["userId", "genres"]],
    on="userId",
    how="left"
)

user_stats.rename(
    columns={"genres": "favorite_genre"},
    inplace=True
)

user_stats.head()

,userId,avg_rating_given,num_movies_rated,favorite_genre
0,1,4.366379,232,Action
1,2,3.948276,29,Drama
2,3,2.435897,39,Drama
3,4,3.555556,216,Comedy
4,5,3.636364,44,Drama


In [33]:
popular_movies = movie_stats[
    movie_stats["num_ratings"] >= 10
]

popular_movies.head()

,movieId,title,avg_rating,num_ratings,rating_std
0,1,Toy Story,3.920930,215,0.834859
1,2,Jumanji,3.431818,110,0.881713
2,3,Grumpier Old Men,3.259615,52,1.054823
4,5,Father of the Bride Part II,3.071429,49,0.907148
5,6,Heat,3.946078,102,0.817224


In [34]:
df = df.merge(
    movie_stats,
    on=["movieId", "title"],
    how="left"
)

df.head()

,userId,movieId,rating,timestamp,title,genres,rating_date,rating_year,rating_month,release_year,avg_rating,num_ratings,rating_std
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,2000-07-30 18:45:03,2000,7,1995.0,3.920930,215,0.834859
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance,2000-07-30 18:20:47,2000,7,1995.0,3.259615,52,1.054823
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller,2000-07-30 18:37:04,2000,7,1995.0,3.946078,102,0.817224
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller,2000-07-30 19:03:35,2000,7,1995.0,3.975369,203,0.922429
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller,2000-07-30 18:48:51,2000,7,1995.0,4.237745,204,0.800921


In [35]:
df = df.merge(
    user_stats,
    on="userId",
    how="left"
)

df.head()

,userId,movieId,rating,timestamp,title,genres,rating_date,rating_year,rating_month,release_year,avg_rating,num_ratings,rating_std,avg_rating_given,num_movies_rated,favorite_genre
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,2000-07-30 18:45:03,2000,7,1995.0,3.920930,215,0.834859,4.366379,232,Action
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance,2000-07-30 18:20:47,2000,7,1995.0,3.259615,52,1.054823,4.366379,232,Action
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller,2000-07-30 18:37:04,2000,7,1995.0,3.946078,102,0.817224,4.366379,232,Action
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller,2000-07-30 19:03:35,2000,7,1995.0,3.975369,203,0.922429,4.366379,232,Action
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller,2000-07-30 18:48:51,2000,7,1995.0,4.237745,204,0.800921,4.366379,232,Action


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   userId            100836 non-null  int64         
 1   movieId           100836 non-null  int64         
 2   rating            100836 non-null  float64       
 3   timestamp         100836 non-null  int64         
 4   title             100836 non-null  object        
 5   genres            100836 non-null  object        
 6   rating_date       100836 non-null  datetime64[ns]
 7   rating_year       100836 non-null  int32         
 8   rating_month      100836 non-null  int32         
 9   release_year      100818 non-null  float64       
 10  avg_rating        100836 non-null  float64       
 11  num_ratings       100836 non-null  int64         
 12  rating_std        97390 non-null   float64       
 13  avg_rating_given  100836 non-null  float64       
 14  num_

In [37]:
df.to_csv(
    "../data/processed/movies_enriched.csv",
    index=False
)

print("Week 2 Completed Successfully!")

Week 2 Completed Successfully!
